Products can have multiple geometric representations.
Each representation belongs to a representation context. This determines how it will be viewed.
Use Types and Materials sets

The below sets up the libraries and other things needed before geometry creation.
 - Measurement units
 - Add contexts and sub contexts

In [377]:
import numpy as np
import ifcopenshell.api.root
import ifcopenshell.api.unit
import ifcopenshell.api.context
import ifcopenshell.api.project
import ifcopenshell.api.geometry

model = ifcopenshell.file()

# You need a project before you can assign units.
ifcopenshell.api.root.create_entity(model, ifc_class="IfcProject")

# Let's say we want coordinates to be in millimeters.
length = ifcopenshell.api.unit.add_si_unit(model, unit_type="LENGTHUNIT", prefix="MILLI")
ifcopenshell.api.unit.assign_unit(model, units=[length])

# Alternatively, you may specify without any arguments to automatically
# create millimeters, square meters, and cubic meters as a convenience for
# testing purposes. Sorry imperial folks, we prioritise metric here.
#ifcopenshell.api.unit.assign_unit(model)



# We want our representation to be the 3D body of the element.
# This representation context is only created once per project.
# You must reuse the same body context every time you create a new representation.
model3d = ifcopenshell.api.context.add_context(model, context_type="Model")
# And/Or, if we plan to store 2D geometry, we need a "Plan" context
plan = ifcopenshell.api.context.add_context(model, context_type="Plan")

body = ifcopenshell.api.context.add_context(model,
    context_type="Model", context_identifier="Body", target_view="MODEL_VIEW", parent=model3d)


# Axis subcontext.
ifcopenshell.api.context.add_context(model,
    context_type="Model", context_identifier="Axis", target_view="GRAPH_VIEW", parent=model3d)

# It's also important to have a 2D Axis subcontext for things like
# walls and claddings which can be drawn using a 2D axis line.
ifcopenshell.api.context.add_context(model,
    context_type="Plan", context_identifier="Axis", target_view="GRAPH_VIEW", parent=plan)

# The 3D Box subcontext is useful for clash detection or shape
# analysis, or even lazy-loading of large models.
ifcopenshell.api.context.add_context(model,
    context_type="Model", context_identifier="Box", target_view="MODEL_VIEW", parent=model3d)

# A 2D annotation subcontext for plan views are important for door
# swings, window cuts, and symbols for equipment like GPOs, fire
# extinguishers, and so on.
ifcopenshell.api.context.add_context(model,
    context_type="Plan", context_identifier="Annotation", target_view="PLAN_VIEW", parent=plan)

#17=IfcGeometricRepresentationSubContext('Annotation','Plan',*,*,*,*,#12,$,.PLAN_VIEW.,$)

In [378]:

# # Create our element with an object placement.
# element = ifcopenshell.api.root.create_entity(model, ifc_class="IfcFurniture")
# ifcopenshell.api.geometry.edit_object_placement(model, product=element)

# # Let's create our representation!
# # See below sections for examples on how to create representations.
# # A wall-like representation, 5 meters long, 3 meters high, and 200mm thick
# representation = ifcopenshell.api.geometry.add_wall_representation(model,
#     context=body, length=5, height=3, thickness=0.2)

# # Assign our new body representation back to our element
# ifcopenshell.api.geometry.assign_representation(model, product=element, representation=representation)


The below code is how to make a transformation matrix to edit the object location

In [379]:
# # Create a 4x4 identity matrix. This matrix is at the origin with no rotation.
# matrix = np.eye(4)
# # Rotate the matix 90 degrees anti-clockwise around the Z axis (i.e. in plan).
# # Anti-clockwise is positive. Clockwise is negative.
# # matrix = ifcopenshell.util.placement.rotation(90, "Z") @ matrix

# # Set the X, Y, Z coordinates. Notice how we rotate first then translate.
# # This is because the rotation origin is always at 0, 0, 0.
# matrix[:,3][0:3] = (0, 4, 0)

In [380]:
# make the first room
R1WN = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")
R1WS = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")

# A wall-like representation, 5 meters long, 3 meters high, and 200mm thick
repW1 = ifcopenshell.api.geometry.add_wall_representation(model,
    context=body, length=5, height=3, thickness=0.2)

repW2 = ifcopenshell.api.geometry.add_wall_representation(model,
    context=body, length=4.2, height=3, thickness=0.2)

matrix = np.eye(4)
matrix[:,3][0:3] = (0, 4, 0)

# Set our wall's Object Placement using our matrix.
# `is_si=True` states that we are using SI units instead of project units.
# ifcopenshell.api.geometry.edit_object_placement(model, product=wall, matrix=matrix, is_si=True)

# make the north and south walls of room 1
ifcopenshell.api.geometry.edit_object_placement(model, product=R1WS, is_si=True)
ifcopenshell.api.geometry.edit_object_placement(model, product=R1WN, matrix=matrix, is_si=True)

# Assign our new body representation back to our element
ifcopenshell.api.geometry.assign_representation(model, product=R1WS, representation=repW1)
ifcopenshell.api.geometry.assign_representation(model, product=R1WN, representation=repW1)

# make the west and south walls of room 1
R1WE = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")
R1WW = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")

matrix = np.eye(4)
matrix = ifcopenshell.util.placement.rotation(90, "Z") @ matrix

ifcopenshell.api.geometry.edit_object_placement(model, product=R1WE, matrix=matrix, is_si=True)
matrix[:,3][0:3] = (5.2, 0, 0)
ifcopenshell.api.geometry.edit_object_placement(model, product=R1WW, matrix=matrix, is_si=True)

ifcopenshell.api.geometry.assign_representation(model, product=R1WE, representation=repW2)
ifcopenshell.api.geometry.assign_representation(model, product=R1WW, representation=repW2)

R1slab = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSlab")

# Create a rectangular profile for the slab with your desired dimensions
profile = model.create_entity("IfcRectangleProfileDef", ProfileName="slab", ProfileType="AREA",
    XDim=5400, YDim=4200)

# A profile-based representation, 1 meter long
repslab = ifcopenshell.api.geometry.add_profile_representation(model, context=body, profile=profile, depth=0.2)

matrix = np.eye(4)
matrix[:,3][0:3] = (2.5, 2.1, -0.2)
ifcopenshell.api.geometry.edit_object_placement(model, product=R1slab, matrix=matrix, is_si=True)
ifcopenshell.api.geometry.assign_representation(model, product=R1slab, representation=repslab)


## make the opening for the door
R1door = ifcopenshell.api.root.create_entity(model, ifc_class="IfcOpeningElement")
doorrep = ifcopenshell.api.geometry.add_wall_representation(model,
    context=body, length=0.9, height=2.1, thickness=0.4)
ifcopenshell.api.geometry.assign_representation(model,
    product=R1door, representation=doorrep)

matrix = np.eye(4)
matrix[:,3] = [1, 4, 0, 0]
ifcopenshell.api.geometry.edit_object_placement(model, product=R1door, matrix=matrix)
ifcopenshell.api.feature.add_feature(model, feature=R1door, element=R1WN)

#98=IfcRelVoidsElement('1S3IpL4krFpv$N3PA3ZnPK',$,$,$,#18,#81)

In [ ]:
# make the second room
R2WN = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")
R2WS = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")

# A wall-like representation, 5 meters long, 3 meters high, and 200mm thick
repW1 = ifcopenshell.api.geometry.add_wall_representation(model,
    context=body, length=8, height=3, thickness=0.2)

repW2 = ifcopenshell.api.geometry.add_wall_representation(model,
    context=body, length=4.2, height=3, thickness=0.2)



# make the north and south walls of room 2
matrix = np.eye(4)
matrix[:,3][0:3] = (5.2, 4, 0)
ifcopenshell.api.geometry.edit_object_placement(model, product=R2WN, matrix=matrix, is_si=True)
matrix[:,3][0:3] = (5.2, 0, 0)
ifcopenshell.api.geometry.edit_object_placement(model, product=R2WS, matrix=matrix, is_si=True)

# Assign our new body representation back to our element
ifcopenshell.api.geometry.assign_representation(model, product=R2WS, representation=repW1)
ifcopenshell.api.geometry.assign_representation(model, product=R2WN, representation=repW1)

# make the east walls of room 2
R2WE = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")

matrix = np.eye(4)
matrix = ifcopenshell.util.placement.rotation(90, "Z") @ matrix
matrix[:,3][0:3] = (13.2, 0, 0)
ifcopenshell.api.geometry.edit_object_placement(model, product=R2WE, matrix=matrix, is_si=True)

ifcopenshell.api.geometry.assign_representation(model, product=R2WE, representation=repW2)

R2slab = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSlab")

# Create a rectangular profile for the slab with your desired dimensions
profile = model.create_entity("IfcRectangleProfileDef", ProfileName="slab", ProfileType="AREA",
    XDim=8000, YDim=4200)

# A profile-based representation, 1 meter long
repslab = ifcopenshell.api.geometry.add_profile_representation(model, context=body, profile=profile, depth=0.2)

matrix = np.eye(4)
matrix[:,3][0:3] = (9.2, 2.1, -0.2)
ifcopenshell.api.geometry.edit_object_placement(model, product=R2slab, matrix=matrix, is_si=True)
ifcopenshell.api.geometry.assign_representation(model, product=R2slab, representation=repslab)


## make the opening for the door
R2door = ifcopenshell.api.root.create_entity(model, ifc_class="IfcOpeningElement")
doorrep = ifcopenshell.api.geometry.add_wall_representation(model,
    context=body, length=0.9, height=2.1, thickness=0.4)
ifcopenshell.api.geometry.assign_representation(model,
    product=R2door, representation=doorrep)

matrix = np.eye(4)
matrix[:,3] = [10, 4, 0, 0]
ifcopenshell.api.geometry.edit_object_placement(model, product=R2door, matrix=matrix)
ifcopenshell.api.feature.add_feature(model, feature=R2door, element=R2WN)

#177=IfcRelVoidsElement('0YB8_Qsyj98RxyT1zIGaYy',$,$,$,#104,#160)

make the corridor below

In [382]:
# make the corridor
CWN = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")

# A wall-like representation, 5 meters long, 3 meters high, and 200mm thick
repW1 = ifcopenshell.api.geometry.add_wall_representation(model,
    context=body, length=13, height=3, thickness=0.2)

repW2 = ifcopenshell.api.geometry.add_wall_representation(model,
    context=body, length=2.2, height=3, thickness=0.2)

matrix = np.eye(4)
matrix[:,3][0:3] = (0, 6.2, 0)

# make the north wall of corridor
ifcopenshell.api.geometry.edit_object_placement(model, product=CWN, matrix=matrix, is_si=True)

# Assign our new body representation back to our element
ifcopenshell.api.geometry.assign_representation(model, product=CWN, representation=repW1)

# make the west and east walls of corridor
CWE = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")
CWW = ifcopenshell.api.root.create_entity(model, ifc_class="IfcWall")

matrix = np.eye(4)
matrix = ifcopenshell.util.placement.rotation(90, "Z") @ matrix
matrix[:,3][0:3] = (0, 4.2, 0)
ifcopenshell.api.geometry.edit_object_placement(model, product=CWW, matrix=matrix, is_si=True)
matrix[:,3][0:3] = (13.2, 4.2, 0)
ifcopenshell.api.geometry.edit_object_placement(model, product=CWE, matrix=matrix, is_si=True)

ifcopenshell.api.geometry.assign_representation(model, product=CWE, representation=repW2)
ifcopenshell.api.geometry.assign_representation(model, product=CWW, representation=repW2)

Cslab = ifcopenshell.api.root.create_entity(model, ifc_class="IfcSlab")

# Create a rectangular profile for the slab with your desired dimensions
profile = model.create_entity("IfcRectangleProfileDef", ProfileName="slab", ProfileType="AREA",
    XDim=13400, YDim=2200)

# A profile-based representation, 1 meter long
repslab = ifcopenshell.api.geometry.add_profile_representation(model, context=body, profile=profile, depth=0.2)

matrix = np.eye(4)
matrix[:,3][0:3] = (6.5, 5.3, -0.2)
ifcopenshell.api.geometry.edit_object_placement(model, product=Cslab, matrix=matrix, is_si=True)
ifcopenshell.api.geometry.assign_representation(model, product=Cslab, representation=repslab)


## make the opening for the corridor exit
Cdoor = ifcopenshell.api.root.create_entity(model, ifc_class="IfcOpeningElement")
doorrep = ifcopenshell.api.geometry.add_wall_representation(model,
    context=body, length=0.9, height=2.1, thickness=0.4)
ifcopenshell.api.geometry.assign_representation(model,
    product=Cdoor, representation=doorrep)

matrix = np.eye(4)
matrix = ifcopenshell.util.placement.rotation(90, "Z") @ matrix
matrix[:,3] = [13.2, 4.8, 0, 0]
ifcopenshell.api.geometry.edit_object_placement(model, product=Cdoor, matrix=matrix)
ifcopenshell.api.feature.add_feature(model, feature=Cdoor, element=CWE)


#256=IfcRelVoidsElement('022JzaGnbB3wO_d10rKiab',$,$,$,#210,#239)

In [383]:
model.write("test.ifc")